In [55]:
from openai import OpenAI

openai_client = OpenAI()

In [56]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)


In [57]:
from minsearch import AppendableIndex

In [58]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)


In [59]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [60]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )


In [61]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [62]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [63]:
print(response.output_text)

The provided context does not contain specific instructions on how to create a dashboard in Evidently.


Making It Agentic

In [64]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}


In [65]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

In [66]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [67]:
tool_call = response.output[0]
tool_call


ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_EjgWaJprWAngTh4H81CmTAOB', name='search', type='function_call', id='fc_0ab452a9bcae09b2006993dbd57948819083bcc38e85e3979d', status='completed')

In [68]:
messages.append(tool_call)

In [69]:
tool_call.arguments

'{"query":"create dashboard in Evidently"}'

In [70]:
arguments = json.loads(tool_call.arguments)
arguments


{'query': 'create dashboard in Evidently'}

In [71]:
search_results = search(query='create dashboard in Evidently')

In [72]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': 'Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\n\n<Check>\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\n</Check>\n\n## Adding Tabs\n\nBy default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them.\n\n**To add a Tab**:\n\n- Enter "Edit" mode on the Dashboard (top right corner).\n- Click the plus sign with "add Tab" on the left.\n- To create a custom Tab, select "empty" and enter a name.\n\nTo simplify setup, you can start with pre-built Tabs. These are dashboard templates with preset Panel combinations:\n\n![Add Dashboard Tab](/images/dashboard/add_dashboard_tab_v2.gif)\n\n**Pre-built Tabs** rely on having related Metrics (or Presets that include the specific Metrics) within the Project. If the necessary data is not available, the Panels will appear empty un

In [73]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [74]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_EjgWaJprWAngTh4H81CmTAOB', name='search', type='function_call', id='fc_0ab452a9bcae09b2006993dbd57948819083bcc38e85e3979d', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_EjgWaJprWAngTh4H81CmTAOB',
  'output': '[{"start": 0, "content": "Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\\n\\n<Check>\\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\\n</Check>\\n\\n## Adding Tabs

In [75]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

4075

In [76]:
print(response.output_text)


To create a dashboard in Evidently, follow these steps:

### 1. **Initial Setup**
- Ensure you have connected to **Evidently Cloud** and created a **Project**.

### 2. **Add Tabs**
You can organize your dashboard using multiple tabs.

- **To add a Tab in UI**:
  - Enter **Edit** mode on the Dashboard (top right corner).
  - Click the **plus sign** next to "Add Tab".
  - Choose "empty" for a custom Tab and enter a name.
  
- **To add a Tab in API**:
  ```python
  project.dashboard.add_tab("Another Tab")
  ```

### 3. **Add Panels**
Panels are used for visualizing results like charts and tables.

- **To add a Panel in UI**:
  - Enter **Edit** mode.
  - Click the **"Add Panel"** button.
  - Configure the panel using the prompts and click **Save**.

- **To add a Panel in API**:
  ```python
  from evidently.sdk.models import PanelMetric
  from evidently.sdk.panels import DashboardPanelPlot

  project.dashboard.add_panel(
      DashboardPanelPlot(
          title="My Dashboard Title",
      

Structured Output

In [77]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [78]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

4299

In [79]:
4299 - 4075

224

In [80]:
rag_response = response.output_parsed
print(rag_response.answer)

## Creating a Dashboard in Evidently

To create a dashboard in Evidently, follow these steps:

### 1. **Create a Project and Connect to Evidently Cloud**
   - Before creating a dashboard, ensure you have connected to [Evidently Cloud](https://docs.evidently.ai/docs/setup/cloud) and created a Project.

### 2. **Add Tabs to Your Dashboard**
You can add multiple tabs to organize your dashboard:
- **To Add a Tab:**  
  1. Enter "Edit" mode on the Dashboard (top right corner).
  2. Click the plus sign to "add Tab" on the left.
  3. Select "empty" to create a custom tab and enter a name.
- **To Delete a Tab:**  
  1. Enter "Edit" mode.
  2. Click "edit Tabs" next to the Tab names on the left, and choose which one to delete.

### 3. **Add Panels to Your Dashboard**
Panels visualize evaluation results.
- **To Add a Panel:**  
  1. Enter "Edit" mode on the Dashboard.
  2. Click the "Add Panel" button.
  3. Follow the prompts to configure the panel, select metrics, and set the visualization type

In [81]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [82]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [83]:
question = "How do I create a dahsbord in Evidently?"

In [84]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [85]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [86]:
message_history.extend(response.output)


In [87]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_tG0PGn0Os2p2NfmWtd058t90', name='search', type='function_call', id='fc_06f49ef6cb042b9f006993dbeb4db08194871e4d121c453988', status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard setup"}', call_id='call_ve6rcX3bCe3B3LeyonAceZOo', name='search', type='function_call', id='fc_06f49ef6cb042b9f006993dbeb4dc4819492d0bdf20c6f2

In [88]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"create dashboard"})...
executing search({"query":"dashboard setup"})...
executing search({"query":"how to build dashboard"})...


In [89]:
len(message_history)

8